In [2]:
import cmath
import math


def fft(x):
    n = len(x)
    if n == 1:
        return [complex(x[0])]

    if n % 2 != 0:
        raise ValueError("potencia de 2")


    pares = fft(x[0::2])
    impares = fft(x[1::2])

    resultado = [0] * n
    for k in range(n // 2):
        angulo = -2j * cmath.pi * k/ n
        twiddle = cmath.exp(angulo) * impares[k]

        resultado[k] = pares[k] + twiddle
        resultado[k + n // 2] = pares[k] - twiddle

    return resultado


def ifft(X):
    n = len(X)
    conjugados = [c.conjugate() for c in X]
    resultado = fft(conjugados)
    return [c.conjugate() / n for c in resultado]


def fft_iterativa(x):
    n = len(x)
    bits = int(math.log2(n))

    X = [complex(x[bit_reversal(i, bits)]) for i in range(n)]

    longitud = 2
    while longitud <= n:
        angulo_base = -2 * cmath.pi/ longitud
        w_base = cmath.exp(1j * angulo_base)


        for i in range(0, n, longitud):
            w = complex(1)
            for k in range(longitud // 2):
                u_val = X[i + k]
                v_val = X[i + k + longitud // 2] * w

                X[i + k] = u_val + v_val
                X[i + k + longitud // 2] = u_val - v_val

                w *= w_base

        longitud *= 2

    return X


def bit_reversal(num, bits):
    resultado = 0
    for _ in range(bits):
        resultado = (resultado << 1) | (num & 1)
        num >>= 1
    return resultado


def redondear(X, decimales = 6):
    return[
        complex(round(c.real, decimales), round(c.imag, decimales))
        for c in X
    ]



if __name__ == "__main__":
    import math



    N = 8
    fs = N
    senal = [
        math.sin(2 * math.pi * 1 * t / N)+
        math.sin(2 * math.pi * 3 * t / N)
        for t in range(N)

    ]


    print("señal original:")
    print([redondear([s],4)[0].real for s in senal]) # Fixed rounds to redondear and extracting real part for print

    espectro = fft(senal)
    print("espectro fft recursiva:")
    print(redondear(espectro))

    recuperada = ifft(espectro) # Defined recuperada
    print("señal recuperada")
    print([round(c.real, 4) for c in recuperada])

    print("magnitudes de espectro:")
    for i, c in enumerate(espectro):
        mag = abs(c)
        if mag > 0.5:
            print(f"frecuencia bin {i}: magnitud = {mag:.4f}")

señal original:
[0.0, 1.4142, 0.0, 1.4142, 0.0, -1.4142, 0.0, -1.4142]
espectro fft recursiva:
[(-0+0j), (-0-4j), 0j, (-0-4j), 0j, (-0+4j), -0j, (-0+4j)]
señal recuperada
[0.0, 1.4142, 0.0, 1.4142, 0.0, -1.4142, 0.0, -1.4142]
magnitudes de espectro:
frecuencia bin 1: magnitud = 4.0000
frecuencia bin 3: magnitud = 4.0000
frecuencia bin 5: magnitud = 4.0000
frecuencia bin 7: magnitud = 4.0000


In [5]:
from collections import defaultdict


class Grafo:
    def __init__(self):
        self.adyacencia = defaultdict(list)

    def agregar_arista(self, u, v):
        self.adyacencia[u].append(v)
        self.adyacencia[v].append(u)

    def __repr__(self):
        return dict(self.adyacencia).__repr__()



def dfs_recursivo(grafo, nodo, visitados=None, orden=None):
    if visitados is None:
        visitados = set()
    if orden is None:
        orden = []

    visitados.add(nodo)
    orden.append(nodo)

    for vecino in grafo.adyacencia[nodo]:
        if vecino not in visitados:
            dfs_recursivo(grafo, vecino, visitados, orden)

    return orden



def dfs_iterativo(grafo, inicio):
    visitados = set()
    pila = [inicio]
    orden = []

    while pila:
        nodo = pila.pop()

        if nodo in visitados:
            continue

        visitados.add(nodo)
        orden.append(nodo)

        for vecino in reversed(grafo.adyacencia[nodo]):
            if vecino not in visitados:
                pila.append(vecino)

    return orden



def dfs_completo(grafo):
    visitados = set()
    componentes = []

    for nodo in list(grafo.adyacencia.keys()): # Iterate over keys to ensure all nodes are checked
        if nodo not in visitados:
            componente_actual = []
            dfs_recursivo(grafo, nodo, visitados, componente_actual)
            componentes.append(componente_actual)

    return componentes


def tiene_ciclo(grafo):
    visitados = set()

    def dfs_ciclo(nodo, padre):
        visitados.add(nodo)

        for vecino in grafo.adyacencia[nodo]:
            if vecino not in visitados:
                if dfs_ciclo(vecino, nodo):
                    return True
            elif vecino != padre:
                return True

        return False


    for nodo in list(grafo.adyacencia.keys()): # Iterate over keys to ensure all nodes are checked
        if nodo not in visitados:
            if dfs_ciclo(nodo, -1):
                return True

    return False


def encontrar_camino(grafo, inicio, fin):
    visitados= set()

    def dfs_camino(nodo, camino):
        if nodo == fin:
            return camino

        visitados.add(nodo)

        for vecino in grafo.adyacencia[nodo]:
            if vecino not in visitados:
                resultado = dfs_camino(vecino, camino + [vecino])
                if resultado:
                    return resultado

        return []


    return dfs_camino(inicio, [inicio]) # Fixed indentation



if __name__ == "__main__":
    g = Grafo()
    aristas = [(1,2),(1,3),(2,4),(2,5),(3,5),(4,5),(4,6),(5,6)]
    for u,v in aristas:
        g.agregar_arista(u,v)

    print("grafo",g)
    print("dfs recursivo desde 1", dfs_recursivo(g,1))
    print("dfs iterativo desde 1:", dfs_iterativo(g,1))
    print("tienen ciclo", tiene_ciclo(g))
    print("camino de 4 a 6:", encontrar_camino(g,4,6)) # Added comma

    g2 = Grafo()
    for u, v in [(1,2),(2,3), (4,5)]:
        g2.agregar_arista(u,v)
    print("componentes conexas:", dfs_completo(g2))

    g3 = Grafo()
    for u, v in [(1,2),(2,3),(3,1)]:
        g3.agregar_arista(u,v)
    print("tiene ciclo g3 : ", tiene_ciclo(g3))

grafo {1: [2, 3], 2: [1, 4, 5], 3: [1, 5], 4: [2, 5, 6], 5: [2, 3, 4, 6], 6: [4, 5]}
dfs recursivo desde 1 [1, 2, 4, 5, 3, 6]
dfs iterativo desde 1: [1, 2, 4, 5, 3, 6]
tienen ciclo True
camino de 4 a 6: [4, 2, 1, 3, 5, 6]
componentes conexas: [[1, 2, 3], [4, 5]]
tiene ciclo g3 :  True
